# Customer Churn Prediction Project

This Jupyter Notebook walks through the entire Machine Learning workflow for predicting customer churn using the **Telco Customer Churn** dataset. Customer churn occurs when customers stop doing business with a company.

### Workflow Steps:
1. **Data Loading & Inspection**
2. **Data Cleaning & Type Conversion**
3. **Exploratory Data Analysis (EDA) & Visualizations**
4. **Data Splitting & Preprocessing Pipeline (Scaling + Encoding)**
5. **Model Training & Comparison (Logistic Regression, Decision Tree, Random Forest, XGBoost)**
6. **Model Evaluation (Accuracy, Precision, Recall, F1-Score, ROC-AUC, Confusion Matrix)**
7. **Saving the Best Pipeline for Streamlit App Deployment**

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
)

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries imported successfully!")

## 2. Load Data

In [ ]:
df_raw = pd.read_csv('data/Telco-Customer-Churn.csv')
print(f"Dataset dimensions: {df_raw.shape}")
df_raw.head()

## 3. Data Cleaning

Let's drop `customerID` (since it is a unique identifier) and clean the `TotalCharges` column, which contains empty spaces representing missing values for new customers (tenure = 0).

In [ ]:
# Drop customer ID
df = df_raw.drop(columns=['customerID']) if 'customerID' in df_raw.columns else df_raw.copy()

# Convert TotalCharges to numeric, coercing spaces to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check for missing values
missing_count = df['TotalCharges'].isnull().sum()
print(f"Missing values in TotalCharges: {missing_count}")

# Fill missing charges with 0.0 (these correspond to tenure = 0)
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

# Map Churn target column to 1 (Yes) and 0 (No)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0}).astype(int)
print("Data cleaning completed. Head of cleaned dataframe:")
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's look at the distribution of the target variable and relations with other features.

In [ ]:
# Target Variable Distribution
plt.figure(figsize=(6, 5))
sns.countplot(x='Churn', data=df, hue='Churn', palette='Set2', legend=False)
plt.title('Distribution of Customer Churn')
plt.xlabel('Churn (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()
print(df['Churn'].value_counts(normalize=True) * 100)

In [ ]:
# Contract Type vs Churn
plt.figure(figsize=(8, 5))
sns.countplot(x='Contract', hue='Churn', data=df, palette='viridis')
plt.title('Churn Count by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Count')
plt.legend(title='Churn', labels=['No', 'Yes'])
plt.show()

In [ ]:
# Continuous Variables distribution grouped by Churn
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Tenure distribution
sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True, ax=axes[0], palette='crest', common_norm=False)
axes[0].set_title('Customer Tenure (Months) vs Churn')
axes[0].set_xlabel('Tenure (Months)')

# Monthly Charges distribution
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, ax=axes[1], palette='flare', common_norm=False)
axes[1].set_title('Monthly Charges vs Churn')
axes[1].set_xlabel('Monthly Charges ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Payment Method vs Churn
plt.figure(figsize=(10, 6))
sns.countplot(y='PaymentMethod', hue='Churn', data=df, palette='Set1')
plt.title('Churn by Payment Method')
plt.xlabel('Count')
plt.ylabel('Payment Method')
plt.legend(title='Churn', labels=['No', 'Yes'])
plt.show()

## 5. Model Preprocessing Pipeline

We will split the data into training and test sets and create a `ColumnTransformer` to handle StandardScaler and OneHotEncoder.

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

# Preprocessor setup
preprocessor = ColumnTransformer( 
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary', sparse_output=False), categorical_cols)
    ]
)
print("Preprocessing pipeline defined successfully.")

## 6. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10),
    'XGBoost': XGBClassifier(random_state=42, n_estimators=100, eval_metric='logloss')
}

model_metrics = {}
best_f1 = 0
best_pipeline = None
best_name = ""

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    model_metrics[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'pipeline': pipeline
    }
    
    if f1 > best_f1:
        best_f1 = f1
        best_pipeline = pipeline
        best_name = name
        
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred))

### Compare Model Metrics

In [ ]:
df_metrics = pd.DataFrame(model_metrics).T.drop(columns=['pipeline'])
print(df_metrics)

# Visualizing model comparison
df_metrics[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].plot(kind='bar', figsize=(12, 6))
plt.title('Comparison of Models on Test Data')
plt.ylabel('Score')
plt.ylim(0.4, 0.9)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()

### Best Model Details & Confusion Matrix

In [ ]:
print(f"Best Model by F1-Score: {best_name} ({best_f1:.4f})")
y_pred_best = best_pipeline.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(cmap='Blues', values_format='d')
plt.title(f'Confusion Matrix - {best_name}')
plt.grid(False)
plt.show()

## 7. Feature Importances (Random Forest)

Let's extract which features are most important from the Random Forest model.

In [ ]:
rf_model = model_metrics['Random Forest']['pipeline'].named_steps['classifier']
cat_encoder = model_metrics['Random Forest']['pipeline'].named_steps['preprocessor'].named_transformers_['cat']

# Get feature names
cat_feature_names = list(cat_encoder.get_feature_names_out(categorical_cols))
all_feature_names = numeric_cols + cat_feature_names

importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

# Create DataFrame for feature importance
df_importance = pd.DataFrame({
    'Feature': [all_feature_names[i] for i in indices],
    'Importance': importances[indices]
})

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=df_importance.head(15), palette='mako')
plt.title('Top 15 Most Important Features (Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.show()

## 8. Exporting the Pipeline

We save our best pipeline so that the web application can run it seamlessly on user inputs.

In [ ]:
os.makedirs('models', exist_ok=True)
model_path = 'models/best_model_pipeline.pkl'
joblib.dump(best_pipeline, model_path)
print(f"Successfully saved the best pipeline model to {model_path}!")